# Master Time Series Analysis & Forecasting

> **Format:** This notebook contains **detailed theory only**. All code cells are empty — **you write every line yourself.**

---


# PART 1 — Foundation

---


## 1.1 Time Series & Datetime Indexing

### What is a Time Series?

A **time series** is a sequence of data points collected at **regular, equally-spaced time intervals**.

$$\{y_1, y_2, y_3, \ldots, y_T\}$$

where $y_t$ is the value observed at time $t$.

**Examples:**
- Monthly airline passengers (1 value per month)
- Daily stock closing price (1 value per day)
- Hourly temperature readings (1 value per hour)
- Quarterly GDP figures (1 value per quarter)

### How is Time Series Different from Normal (Tabular) Data?

In normal tabular data (like a CSV of house prices), each row is **independent** — you can shuffle the rows and nothing changes. In time series, **order matters**. January comes before February. Today's stock price is influenced by yesterday's.

| Aspect | Time Series | Tabular (Cross-Sectional) |
|--------|------------|---------------------------|
| **Order** | Matters (temporal) | Doesn't matter |
| **Rows** | Correlated with neighbors | Independent (i.i.d.) |
| **Train/Test** | Must split by time | Random split OK |
| **Goal** | Forecast future | Predict target variable |

### Datetime Indexing in Pandas

For Pandas to treat a DataFrame as time series, the **index must be a `DatetimeIndex`**. This unlocks time-aware operations like `.resample()`, `.shift()`, `.rolling()`, `.diff()`, etc.

**Key concepts:**
- `pd.to_datetime()` — converts strings to datetime objects
- `pd.date_range()` — generates a sequence of dates at a given frequency
- `df.set_index('date_column')` — makes a column the DatetimeIndex
- **Frequency codes**: `'D'` (daily), `'W'` (weekly), `'MS'` (month start), `'YE'` (year end), `'h'` (hourly)

Once the index is a DatetimeIndex, you can slice by date strings: `df['2020-01':'2020-06']`

---


In [ ]:
# 1.1 — Create a DatetimeIndex and a simple DataFrame


## 1.2 Trend

**Trend** ($T_t$) is the **long-term directional movement** in the data.

- **Upward trend**: Values generally increase over time (e.g., population growth, GDP)
- **Downward trend**: Values generally decrease (e.g., declining newspaper sales)
- **Flat/No trend**: Values oscillate around a constant mean

### Types of Trend

| Type | Formula | Example |
|------|---------|--------|
| **Linear** | $T_t = a + bt$ | Steady growth/decline |
| **Polynomial** | $T_t = a + bt + ct^2$ | Accelerating growth |
| **Exponential** | $T_t = ae^{bt}$ | Compound growth |
| **Piecewise** | Changes slope at breakpoints | Different growth phases |

### How to Estimate Trend

1. **Moving Average**: Smooth the series using a centered moving average with window = seasonal period. This averages out the seasonal ups and downs, leaving just the trend.
2. **Linear Regression**: Fit a straight line $y = a + bt$ to the data.
3. **LOESS (Locally Weighted Scatterplot Smoothing)**: Fits local polynomials — more flexible than a single line.

### Why Trend Matters

A trending series has a **changing mean** over time. This means it's **non-stationary** — and most classical models (ARIMA) require stationarity. So we often need to **remove the trend** before modeling (via differencing or detrending).

---


In [ ]:
# 1.2 — Generate data with a clear trend and plot it


## 1.3 Seasonality

**Seasonality** ($S_t$) is a **fixed-period, repeating pattern** in the data.

The key word is **fixed period** — it repeats at the same interval every time:
- Retail sales spike every December (period = 12 months)
- Restaurant traffic peaks every Friday-Saturday (period = 7 days)
- Electricity demand peaks every afternoon (period = 24 hours)

### Properties of Seasonality

- **Period is known** — you know it's 12 months, 7 days, or 24 hours
- **Pattern is predictable** — same months/days/hours show same behavior year after year
- **Caused by external calendar factors** — weather, holidays, work schedules, school terms

### Detecting Seasonality

1. **Visual inspection**: Plot the data and look for repeating waves
2. **Subseries plot**: Plot each month (or day/hour) separately across years — if each month's values cluster together, there's seasonality
3. **ACF plot**: Significant spikes at seasonal lags ($m, 2m, 3m, \ldots$)
4. **Decomposition**: Use `seasonal_decompose()` or `STL()` to extract the seasonal component

### Seasonal Amplitude

The **size of seasonal swings** can be:
- **Constant** — same absolute magnitude every year (→ additive model)
- **Proportional** — grows/shrinks with the trend (→ multiplicative model)

---


In [ ]:
# 1.3 — Generate and plot data with clear seasonality


## 1.4 Cyclic vs Seasonal Pattern

Students often confuse **cyclical** and **seasonal** patterns. They look similar (both are waves) but differ in key ways:

| | Seasonal | Cyclical |
|--|----------|----------|
| **Period** | Fixed and known (12 months, 7 days) | Variable and unknown (2-10 years) |
| **Cause** | Calendar factors (weather, holidays) | Economic/business dynamics |
| **Predictable** | Yes (same time every year) | Not easily (duration varies) |
| **Example** | Ice cream sales peak in July | Economic boom/recession cycles |
| **In models** | Captured by seasonal terms (P, D, Q) | Captured by AR terms |

### Key Insight

- If you can say "this peak happens every July" → **Seasonal**
- If you can say "there are waves but I don't know when the next peak will come" → **Cyclical**

In practice, **cyclical** patterns are often grouped with the **trend** into a single **Trend-Cycle** component, because neither has a fixed repeating period.

---


In [ ]:
# 1.4 — Create data with cyclical behavior (no fixed period) and compare to seasonal


## 1.5 Noise / Residual

**Residual** ($R_t$ or $\epsilon_t$) is everything that remains after you remove Trend, Seasonality, and Cyclical components.

$$R_t = Y_t - T_t - S_t - C_t \quad \text{(additive)}$$

### Properties of Good Residuals

If your model is capturing all the structure in the data, residuals should be **white noise**:

1. **Mean = 0** — no systematic bias
2. **Constant variance** — no patterns in spread
3. **No autocorrelation** — residual at time $t$ is uncorrelated with residual at time $t-1$
4. **Normally distributed** (ideally, for prediction intervals)

### Why Residuals Matter

- If residuals have **patterns** (e.g., ACF shows significant spikes) → your model missed some structure → go back and improve it
- If residuals are **white noise** → your model has captured all useful information → good!

Residual analysis is a key step in the **Box-Jenkins methodology** (Part 5).

---


In [ ]:
# 1.5 — Generate white noise and verify its properties (mean, std, plot)


## 1.6 Additive vs Multiplicative Model

### Additive Model

$$\boxed{Y_t = T_t + S_t + R_t}$$

**When to use:** Seasonal fluctuations have **constant absolute magnitude** regardless of the level.

Visually: The seasonal bands stay the **same width** over time.

Example: Temperature varies by ±10°C every year — this doesn't change whether it's 1980 or 2020.

### Multiplicative Model

$$\boxed{Y_t = T_t \times S_t \times R_t}$$

**When to use:** Seasonal fluctuations **scale proportionally** with the level.

Visually: The seasonal bands **widen** as the trend increases.

Example: Airline passengers — as traffic grows from 100K to 500K, summer peaks grow from ±10K to ±50K.

### How to Choose?

1. **Visual test**: Plot the series. Are the seasonal waves getting wider? → Multiplicative
2. **Residual variance**: Decompose with both models. The one with **lower residual variance** fits better.
3. **Log trick**: If data is multiplicative, taking $\log(Y_t)$ converts it to additive:
   $$\log(T_t \times S_t \times R_t) = \log(T_t) + \log(S_t) + \log(R_t)$$
4. **Data with zeros or negatives**: Must use additive (can't take log of 0 or negative).

---


In [ ]:
# 1.6 — Build additive and multiplicative series, plot side-by-side


## 1.7 Classical Decomposition

Classical decomposition splits a time series into **Trend + Seasonal + Residual** using moving averages.

### Algorithm (Additive)

**Step 1 — Estimate Trend** ($\hat{T}_t$):
Apply a **centered moving average** of order $m$ (= seasonal period).
- For odd $m$: Simple centered MA
- For even $m$ (e.g., 12): Take a $2 \times m$-MA (MA of two overlapping MAs) to center it properly

**Step 2 — Detrend**:
$$D_t = Y_t - \hat{T}_t$$

**Step 3 — Estimate Seasonality** ($\hat{S}_t$):
For each season (e.g., each month 1-12), take the **average of all detrended values** for that season. This gives 12 seasonal indices. Then adjust so they sum to zero (additive) or average to 1 (multiplicative).

**Step 4 — Residual**:
$$\hat{R}_t = Y_t - \hat{T}_t - \hat{S}_t$$

### Limitations

- **Missing values at boundaries**: First and last $m/2$ trend values are `NaN`
- **Fixed seasonality**: Assumes the seasonal pattern is exactly the same every year (never evolves)
- **Sensitive to outliers**: One extreme value affects the entire seasonal estimate
- **Only additive or multiplicative**: Can't handle mixed models

### Python
`from statsmodels.tsa.seasonal import seasonal_decompose`

---


In [ ]:
# 1.7 — Use seasonal_decompose() on your data and plot all components


## 1.8 STL Decomposition

**STL** = Seasonal and Trend decomposition using **Loess** (LOcally Estimated Scatterplot Smoothing).

STL is a **significant upgrade** over classical decomposition:

| Feature | Classical | STL |
|---------|-----------|-----|
| Seasonal pattern | Fixed (never changes) | **Can evolve** over time |
| Trend smoothness | Limited control | Controllable via `trend` parameter |
| Outlier handling | Sensitive | **Robust mode** available |
| Boundary values | Missing | **No missing values** |
| Flexibility | Low | High |

### How STL Works (Conceptual)

STL uses an **iterative algorithm** with two loops:
1. **Inner loop**: Alternates between extracting the seasonal component (using LOESS smoothing on subseries) and the trend component (using LOESS smoothing on deseasonalized data)
2. **Outer loop** (robust mode): Calculates weights that downweight outliers, then reruns the inner loop

### Key Parameters

| Parameter | What it controls |
|-----------|------------------|
| `period` | Seasonal period (must specify correctly) |
| `seasonal` | LOESS window for seasonal extraction (larger = more rigid seasonal pattern) |
| `trend` | LOESS window for trend extraction (larger = smoother trend) |
| `robust` | If `True`, downweights outliers |

### Python
`from statsmodels.tsa.seasonal import STL`

---


In [ ]:
# 1.8 — Run STL decomposition and compare it to classical decomposition


## 1.9 Multiple / Complex Seasonality

Some time series have **more than one seasonal pattern** at different frequencies.

### Examples

| Data Frequency | Seasonal Pattern 1 | Seasonal Pattern 2 | Seasonal Pattern 3 |
|----------------|--------------------|--------------------|--------------------|
| **Hourly** electricity | Daily (24h) | Weekly (168h) | Yearly (8760h) |
| **Daily** retail sales | Weekly (7d) | Yearly (365d) | — |
| **Sub-daily** web traffic | Daily (24h) | Weekly (168h) | — |

### Why This is a Problem

- Standard `seasonal_decompose()` and SARIMA handle **only one** seasonal period
- If your data has daily + yearly seasonality, SARIMA with $m=365$ has too many parameters

### Solutions

1. **MSTL** (Multiple STL): Extension of STL that handles multiple seasonal periods
   - `from statsmodels.tsa.seasonal import MSTL`
2. **Fourier terms**: Add sin/cos features at different frequencies as exogenous variables in SARIMAX
3. **Prophet**: Handles multiple seasonalities natively using Fourier series
4. **Deep Learning** (N-BEATS, TFT): Learn complex patterns automatically

### MSTL Concept

$$Y_t = T_t + S_t^{(1)} + S_t^{(2)} + \ldots + S_t^{(K)} + R_t$$

Each $S_t^{(k)}$ captures one seasonal component at period $m_k$.

---


In [ ]:
# 1.9 — Generate data with two seasonal patterns and attempt to decompose


# PART 2 — Statistics & Stationarity

---


## 2.1 Mean & Variance Over Time

The simplest way to check if a series is stationary is to look at how its **mean** and **variance** change over time.

### Rolling Statistics

Compute the mean and standard deviation over a **sliding window** (e.g., last 12 months) and plot them over time:

- **Rolling Mean** = $\frac{1}{w} \sum_{i=0}^{w-1} y_{t-i}$ (average of last $w$ values)
- **Rolling Std** = standard deviation of last $w$ values

### What to Look For

| Rolling Stat Behavior | Interpretation |
|----------------------|----------------|
| Mean is **flat** | Mean is constant → good |
| Mean is **rising/falling** | Trend exists → non-stationary |
| Std is **flat** | Variance is constant → good |
| Std is **rising** | Variance is growing → non-stationary (consider log transform) |

This is a **visual/informal** check. For formal testing, we use ADF and KPSS (coming next).

---


In [ ]:
# 2.1 — Plot rolling mean and rolling std for your time series


## 2.2 Stationarity Intuition

A series is **stationary** if its **statistical properties don't change over time**.

### Three Conditions for Weak (Wide-Sense) Stationarity

**1. Constant Mean:**
$$E[Y_t] = \mu \quad \text{for all } t$$
The average level doesn't drift up or down.

**2. Constant Variance:**
$$\text{Var}(Y_t) = \sigma^2 \quad \text{for all } t$$
The spread doesn't grow or shrink.

**3. Autocovariance Depends Only on Lag, Not on Time:**
$$\text{Cov}(Y_t, Y_{t+h}) = \gamma(h) \quad \text{for all } t$$
The correlation between two points depends only on the **distance** between them, not **when** they occur.

### Intuitive Analogy

Imagine taking a photo of the series through a window of fixed width. If you slide that window anywhere along the timeline and the "statistical snapshot" always looks the same — same average height, same spread, same wiggly pattern — the series is stationary.

### Strict vs Weak Stationarity

- **Strict stationarity**: The entire joint probability distribution is time-invariant. Very strong, rarely tested.
- **Weak stationarity**: Only the first two moments (mean, variance) and autocovariance are time-invariant. This is what we usually mean and what ADF/KPSS test.

---


## 2.3 Why Stationarity Matters

### For Modeling

AR, MA, ARMA, ARIMA — all these models **assume stationarity** (after differencing). If you feed non-stationary data:
- Parameter estimates become **unreliable**
- Forecasts can **diverge** wildly
- Statistical tests (t-tests, F-tests) give **meaningless** results (spurious regression)

### Spurious Regression

If you regress two independent non-stationary series against each other, you'll often get a **high R² and significant coefficients** even though there's no real relationship. This is because both are trending and appear correlated by coincidence.

### Unit Root

Consider the process: $y_t = \phi \cdot y_{t-1} + \epsilon_t$

| $\phi$ value | Behavior |
|-------------|----------|
| $|\phi| < 1$ | **Stationary** — shocks die out, series reverts to mean |
| $\phi = 1$ | **Unit root (Random Walk)** — shocks persist forever, variance → ∞ |
| $|\phi| > 1$ | **Explosive** — series diverges to ±∞ |

When $\phi = 1$: $y_t = y_{t-1} + \epsilon_t$ (Random Walk). The variance grows linearly: $\text{Var}(y_t) = t \cdot \sigma^2_\epsilon$. This is the classic non-stationary process.

---


## 2.4 Augmented Dickey-Fuller (ADF) Test

The most popular formal test for a **unit root** (non-stationarity).

### Model

$$\Delta y_t = \alpha + \beta t + \gamma y_{t-1} + \sum_{i=1}^{p} \delta_i \Delta y_{t-i} + \epsilon_t$$

| Term | Purpose |
|------|--------|
| $\alpha$ | Drift / constant |
| $\beta t$ | Deterministic time trend |
| $\gamma y_{t-1}$ | **The unit root test variable** |
| $\sum \delta_i \Delta y_{t-i}$ | Lagged differences to absorb autocorrelation |

### Hypotheses

- **$H_0: \gamma = 0$** → Unit root exists → **NON-STATIONARY**
- **$H_1: \gamma < 0$** → No unit root → **STATIONARY**

### Decision Rule

| p-value | Action |
|---------|--------|
| $p \leq 0.05$ | **Reject $H_0$** → Series is stationary |
| $p > 0.05$ | **Fail to reject $H_0$** → Series is non-stationary |

The number of lagged terms $p$ is usually chosen automatically via **AIC** (`autolag='AIC'`).

### Python
`from statsmodels.tsa.stattools import adfuller`

---


In [ ]:
# 2.4 — Run ADF test on raw and differenced data


## 2.5 KPSS Test (Kwiatkowski-Phillips-Schmidt-Shin)

KPSS has the **opposite null hypothesis** from ADF:

- **$H_0$**: Series **IS stationary** (this is the opposite of ADF!)
- **$H_1$**: Series is **NOT stationary**

### Decision Rule

| p-value | Action |
|---------|--------|
| $p > 0.05$ | **Fail to reject $H_0$** → STATIONARY |
| $p \leq 0.05$ | **Reject $H_0$** → NON-STATIONARY |

### Two Variants

| `regression=` | Tests for |
|---------------|----------|
| `'c'` | Level stationarity (constant mean) |
| `'ct'` | Trend stationarity (stationary around a deterministic trend) |

### ADF + KPSS Decision Matrix

Using **both tests together** gives a robust diagnosis:

| ADF | KPSS | Conclusion |
|-----|------|------------|
| Stationary (p≤0.05) | Stationary (p>0.05) | **Stationary** ✓ |
| Non-stationary | Non-stationary | **Non-Stationary** → Difference |
| Stationary | Non-stationary | **Trend-Stationary** → Remove deterministic trend |
| Non-stationary | Stationary | **Inconclusive** → More data or different lags |

### Python
`from statsmodels.tsa.stattools import kpss`

---


In [ ]:
# 2.5 — Run KPSS test and combine with ADF for diagnosis


## 2.6 Differencing

**Differencing** is the primary tool to make a non-stationary series stationary.

### First-Order Differencing ($d=1$)

$$\Delta y_t = y_t - y_{t-1}$$

Removes **linear trend**. This is the most common transformation.

### Second-Order Differencing ($d=2$)

$$\Delta^2 y_t = \Delta(\Delta y_t) = y_t - 2y_{t-1} + y_{t-2}$$

Removes **quadratic trend**. Rarely needed. If you need $d > 2$, reconsider your approach.

### Seasonal Differencing ($D=1$)

$$\Delta_m y_t = y_t - y_{t-m}$$

where $m$ = seasonal period. Removes **seasonal patterns** (compare each point to the same season last year).

### Combined Differencing

For data with trend AND seasonality: apply seasonal differencing first, then regular differencing.

### Over-Differencing Warning

**Check variance after each differencing step.** Variance should **decrease**. If it increases → you've over-differenced and introduced artificial patterns.

---


In [ ]:
# 2.6 — Apply d=1, D=1, and both. Compare variances


## 2.7 Log & Box-Cox Transformations

### Problem: Non-Constant Variance

Differencing handles changing **mean** (trend). But what if **variance** is also changing (seasonal bands widen)? Differencing alone won't fix this.

### Log Transform

$$z_t = \ln(y_t)$$

- Compresses large values, expands small values → **stabilizes variance**
- Converts multiplicative seasonality to additive
- Requires all $y_t > 0$

### Box-Cox Transform (Generalized)

$$z_t = \begin{cases} \frac{y_t^\lambda - 1}{\lambda} & \text{if } \lambda \neq 0 \\ \ln(y_t) & \text{if } \lambda = 0 \end{cases}$$

| $\lambda$ | Transform | When to use |
|-----------|----------|-------------|
| 1.0 | Identity (no transform) | Variance already constant |
| 0.5 | Square root | Moderate variance increase |
| 0.0 | Log | Strong variance increase |
| -1.0 | Reciprocal | Very strong variance increase |

The optimal $\lambda$ is found via **maximum likelihood**. `scipy.stats.boxcox()` does this automatically.

### Typical Workflow

1. Check if variance changes → if yes, apply log or Box-Cox
2. Check if mean changes → if yes, apply differencing
3. Result: stationary series ready for modeling

---


In [ ]:
# 2.7 — Apply log and Box-Cox, plot results


# PART 3 — Time-Series Feature Engineering

---


## 3.1 Lag Features

A **lag feature** uses a **past value** of the target as a predictor:

$$\text{lag\_1}[t] = y_{t-1}, \quad \text{lag\_12}[t] = y_{t-12}$$

This is how we convert a time series into a **supervised learning problem** — the target is $y_t$ and the features are $y_{t-1}, y_{t-2}, \ldots$

### Choosing Which Lags

- **lag_1**: Always useful (yesterday's value predicts today)
- **lag_m** (seasonal lag): Very useful (same month last year)
- **ACF/PACF**: Use these plots to identify which lags have significant correlation

### Important: Data Leakage Risk

When creating lag features, make sure you **only use past data**. If lag_1 at time $t$ accidentally uses $y_t$ instead of $y_{t-1}$, you have **leakage** and your model will appear unrealistically good.

### Python
`df['lag_1'] = df['y'].shift(1)`

---


In [ ]:
# 3.1 — Create lag features and check their correlation with target


## 3.2 Rolling Statistics

Compute statistics over a **sliding window** of the last $w$ observations:

| Feature | What it captures |
|---------|------------------|
| Rolling Mean | Local average / smoothed trend |
| Rolling Std | Local volatility |
| Rolling Min/Max | Local support/resistance levels |
| Rolling Median | Robust central tendency |

### EWMA (Exponentially Weighted Moving Average)

$$\text{EWMA}_t = \alpha \cdot y_t + (1 - \alpha) \cdot \text{EWMA}_{t-1}$$

Gives **more weight to recent** observations (unlike simple rolling which weights equally).

### Important: Shift Before Rolling!

When using rolling features for prediction, you must **shift by 1** before computing:
`df['roll_mean'] = df['y'].shift(1).rolling(7).mean()`

Otherwise, the rolling mean at time $t$ includes $y_t$ itself → **data leakage**.

---


In [ ]:
# 3.2 — Create rolling mean, rolling std, and EWMA features


## 3.3 Expanding Statistics

Unlike rolling (fixed window), expanding includes **all data from the start up to time $t$**:

$$\text{expanding\_mean}(t) = \frac{1}{t} \sum_{i=1}^{t} y_i$$

| | Rolling | Expanding |
|--|---------|----------|
| Window size | Fixed ($w$) | Grows (1, 2, 3, ..., $t$) |
| Memory | Forgets old data | Remembers everything |
| Stability | Responsive | Very stable |

### Python
`df['expanding_mean'] = df['y'].expanding().mean()`

---


In [ ]:
# 3.3 — Create expanding mean and compare to rolling mean


## 3.4 Calendar / Time Features

Extract **temporal attributes** from the datetime index:

| Feature | Values | Captures |
|---------|--------|----------|
| `month` | 1-12 | Monthly seasonality |
| `day_of_week` | 0-6 | Weekly patterns |
| `quarter` | 1-4 | Quarterly patterns |
| `is_weekend` | 0/1 | Weekday vs weekend |
| `hour` | 0-23 | Intra-day patterns |

### Cyclical Encoding (Sin/Cos)

Problem: Month 12 (December) and Month 1 (January) are **neighbors**, but numerically 12 and 1 are far apart. This confuses ML models.

Solution: Encode cyclical features using sin and cos:

$$x_{\sin} = \sin\left(\frac{2\pi \cdot v}{\text{max}}\right) \qquad x_{\cos} = \cos\left(\frac{2\pi \cdot v}{\text{max}}\right)$$

Now December (sin, cos) and January (sin, cos) are **close** in the 2D sin-cos space.

---


In [ ]:
# 3.4 — Extract month, quarter, sin/cos features


## 3.5 Time-Series → Supervised Learning

To use ML algorithms (XGBoost, Random Forest), you must convert the time series into a **tabular dataset**:

```
Original:  [y1, y2, y3, y4, y5, y6]

Supervised Table (lag=2):
| lag_2 | lag_1 | target |
|-------|-------|--------|
|  y1   |  y2   |  y3    |
|  y2   |  y3   |  y4    |
|  y3   |  y4   |  y5    |
|  y4   |  y5   |  y6    |
```

Each row is: **features** = past values → **target** = current/future value.

You can add rolling stats, calendar features, and exogenous variables as additional columns.

---


In [ ]:
# 3.5 — Create a full supervised learning table from your time series


## 3.6 Time-Based Train/Test Split

**NEVER use random train/test split for time series.**

Why? Random split puts future data in the training set → your model "sees the future" → overly optimistic scores → model fails in production.

### Correct Approach

Split at a **time cutoff**: everything before the cutoff is training, everything after is testing.

```
[=========TRAIN=========][====TEST====]
past ──────────────────────────────> future
```

---


In [ ]:
# 3.6 — Split your data temporally into train and test


## 3.7 Walk-Forward Validation / TimeSeriesSplit

A single train/test split can be misleading. **Walk-Forward CV** gives a more robust estimate by using multiple folds:

```
Fold 1: [TRAIN====] [TEST=]
Fold 2: [TRAIN======] [TEST=]
Fold 3: [TRAIN========] [TEST=]
Fold 4: [TRAIN==========] [TEST=]
```

- Training set **expands** with each fold (or slides with fixed size)
- Test set always comes **after** training set
- Mimics real-world: train on all available history → predict next period

### Sliding Window Variant

Training window has **fixed size** (slides forward). Better when old data becomes irrelevant.

### Python
`from sklearn.model_selection import TimeSeriesSplit`
- `max_train_size` = set for sliding window
- `gap` = add gap between train and test (for lag features)

---


In [ ]:
# 3.7 — Visualize TimeSeriesSplit folds


# PART 4 — Dependence & Diagnostics

---


## 4.1 Autocorrelation

**Autocorrelation** = correlation of a series with **its own lagged values**.

$$\rho(h) = \frac{\text{Cov}(Y_t, Y_{t+h})}{\sqrt{\text{Var}(Y_t) \cdot \text{Var}(Y_{t+h})}} = \frac{\gamma(h)}{\gamma(0)}$$

- $\rho(0) = 1$ always (series correlated perfectly with itself)
- $\rho(h)$ ranges from $-1$ to $+1$
- High $\rho(1)$ means today's value strongly predicts tomorrow's
- High $\rho(12)$ means January this year is similar to January last year

---


## 4.2 ACF & PACF

### ACF (Autocorrelation Function)

Plots $\rho(h)$ for lags $h = 0, 1, 2, \ldots$

ACF measures **total** correlation at lag $h$, including **indirect** effects through intermediate lags.

### PACF (Partial Autocorrelation Function)

Measures **direct** correlation at lag $h$, with the effects of all intermediate lags **removed**.

$$\phi_{hh} = \text{Corr}(Y_t, Y_{t-h} \mid Y_{t-1}, \ldots, Y_{t-h+1})$$

### How to Read ACF/PACF

- **Blue shaded band** = 95% confidence interval ($\approx \pm 1.96/\sqrt{n}$)
- Bars **outside** the band = statistically significant
- "Cuts off" = drops to zero suddenly after lag $k$
- "Decays" = gradually tapers toward zero

### Pattern Identification (CRITICAL for ARIMA!)

| ACF | PACF | Model |
|-----|------|-------|
| Decays gradually | **Cuts off at lag p** | **AR(p)** |
| **Cuts off at lag q** | Decays gradually | **MA(q)** |
| Both decay | Both decay | **ARMA(p,q)** — try small p, q |
| Slow linear decay | — | **Non-stationary** — difference first! |
| Spikes at $m, 2m, 3m$ | — | **Seasonal** component needed |

---


In [ ]:
# 4.2 — Plot ACF and PACF for raw and differenced series


## 4.3 Ljung-Box Test

A formal test for whether a group of autocorrelations are **significantly different from zero**.

### Hypotheses

- **$H_0$**: The data are independently distributed (no autocorrelation) — i.e., **residuals are white noise**
- **$H_1$**: The data exhibit serial correlation — residuals have **patterns**

### Test Statistic

$$Q(H) = n(n+2) \sum_{h=1}^{H} \frac{\hat{\rho}^2(h)}{n-h}$$

Under $H_0$, $Q \sim \chi^2_{H-p-q}$ where $p, q$ are the ARIMA orders.

### Decision

- $p > 0.05$ → Fail to reject $H_0$ → Residuals are white noise → **Model is adequate**
- $p \leq 0.05$ → Reject $H_0$ → Residuals have patterns → **Model needs improvement**

Typically test at multiple lag groups (e.g., $H = 6, 12, 18, 24$).

### Python
`from statsmodels.stats.diagnostic import acorr_ljungbox`

---


In [ ]:
# 4.3 — Run Ljung-Box test on model residuals


## 4.4 Residual Diagnostics

After fitting a model, always check residuals. A good model's residuals should be **white noise**.

### 4 Diagnostic Plots

1. **Residuals over time** — should look like random noise centered at 0, no patterns
2. **Histogram** — should be roughly bell-shaped (normal distribution)
3. **ACF of residuals** — all bars should be within the confidence band (no autocorrelation)
4. **Q-Q Plot** — points should lie on the diagonal line (normality check)

### Formal Tests

- **Ljung-Box** (above): Tests for autocorrelation in residuals
- **Shapiro-Wilk**: Tests for normality of residuals
- **Heteroscedasticity check**: Plot residuals vs time — variance should be constant

If diagnostics **fail**: go back and try different model orders, add seasonal terms, or apply transformations.

---


In [ ]:
# 4.4 — Create a 4-panel diagnostic plot for model residuals


## 4.5 Outliers & Anomaly Detection

**Outliers** are observations that deviate significantly from the expected pattern.

### Types in Time Series

| Type | Description | Impact |
|------|-------------|--------|
| **Additive Outlier** | Single point spike, returns to normal | Affects one observation |
| **Innovative Outlier** | Shock that propagates through time | Affects all subsequent values |
| **Level Shift** | Permanent change in level | Changes mean permanently |
| **Temporary Change** | Spike that decays back gradually | Affects several observations |

### Detection Methods

1. **Z-score on residuals**: Flag where $|z| > 3$
2. **IQR method**: Flag where value < Q1 - 1.5·IQR or > Q3 + 1.5·IQR
3. **STL + residual check**: Decompose, then flag large residuals
4. **Rolling statistics**: Flag where value deviates > $k$ standard deviations from rolling mean

### Handling Outliers

- Use **STL with `robust=True`** — automatically downweights outliers
- **Winsorize** — cap extreme values at a percentile
- **Interpolate** — replace outliers with interpolated values
- **Leave them** — if they represent genuine events (e.g., COVID crash)

---


In [ ]:
# 4.5 — Detect outliers using z-score on residuals


# PART 5 — Classical Forecasting

---


## 5.1 Naive & Seasonal-Naive Baselines

**Always start with a baseline.** If your fancy model can't beat a simple baseline, it's useless.

### Naive Forecast

$$\hat{y}_{t+h} = y_t \quad \text{(last observed value repeated)}$$

"Tomorrow will be the same as today."

### Seasonal Naive Forecast

$$\hat{y}_{t+h} = y_{t+h-m} \quad \text{(same season last year)}$$

"This January will be like last January."

### Average Forecast

$$\hat{y}_{t+h} = \bar{y} \quad \text{(historical mean)}$$

"Predict the overall average."

These baselines are **surprisingly hard to beat** in many real-world scenarios, especially seasonal naive.

---


In [ ]:
# 5.1 — Implement naive and seasonal naive forecasts, compute MAE


## 5.2 MAE & RMSE Introduction

You need metrics to evaluate forecasts. We introduce two basic ones here (more in Part 7).

### MAE (Mean Absolute Error)

$$\text{MAE} = \frac{1}{n} \sum_{t=1}^{n} |y_t - \hat{y}_t|$$

- Easy to interpret: "on average, my forecast is off by X units"
- Treats all errors equally

### RMSE (Root Mean Squared Error)

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{t=1}^{n} (y_t - \hat{y}_t)^2}$$

- **Penalizes large errors more** (because of squaring)
- Always $\geq$ MAE
- If RMSE >> MAE, you have some very large errors

---


In [ ]:
# 5.2 — Compute MAE and RMSE for your baseline forecasts


## 5.3 Autoregressive Model — AR(p)

$$\boxed{Y_t = c + \phi_1 Y_{t-1} + \phi_2 Y_{t-2} + \ldots + \phi_p Y_{t-p} + \epsilon_t}$$

The current value is a **linear combination of its own past values** (plus noise).

- $p$ = number of past values used (the "order")
- $\phi_1, \ldots, \phi_p$ = coefficients (learned from data)
- **Stationarity condition**: Roots of the characteristic polynomial must lie outside the unit circle. For AR(1): $|\phi_1| < 1$.

### AR(1) Properties
| Property | Value |
|----------|-------|
| Mean | $\mu = c / (1 - \phi_1)$ |
| ACF | Decays exponentially: $\rho(h) = \phi_1^h$ |
| PACF | **Cuts off** after lag 1 |

### How to choose $p$?
Look at PACF — it **cuts off** after lag $p$.

---


In [ ]:
# 5.3 — Fit an AR model and examine coefficients


## 5.4 Moving Average Model — MA(q)

$$\boxed{Y_t = \mu + \epsilon_t + \theta_1 \epsilon_{t-1} + \theta_2 \epsilon_{t-2} + \ldots + \theta_q \epsilon_{t-q}}$$

The current value depends on **past error terms** (not past values).

- $q$ = number of past errors used
- $\theta_1, \ldots, \theta_q$ = coefficients
- **Always stationary** (for any finite $q$)
- **Finite memory**: MA(q) only remembers last $q$ shocks. Forecast converges to the mean after $q$ steps.

### MA(1) Properties
| Property | Value |
|----------|-------|
| ACF | **Cuts off** after lag 1: $\rho(1) = \theta_1/(1+\theta_1^2)$, $\rho(h) = 0$ for $h > 1$ |
| PACF | Decays exponentially |

### How to choose $q$?
Look at ACF — it **cuts off** after lag $q$.

### AR vs MA Summary

| | AR(p) | MA(q) |
|--|-------|-------|
| Depends on | Past **values** | Past **errors** |
| ACF | Decays | **Cuts off** |
| PACF | **Cuts off** | Decays |
| Memory | Infinite (recursive) | Finite ($q$ steps) |
| Stationarity | Needs $|\phi| < 1$ | **Always** stationary |

---


In [ ]:
# 5.4 — Fit an MA model and compare with AR


## 5.5 ARMA(p, q) Concept

$$\boxed{Y_t = c + \sum_{i=1}^{p} \phi_i Y_{t-i} + \epsilon_t + \sum_{j=1}^{q} \theta_j \epsilon_{t-j}}$$

Combines AR and MA. Both ACF and PACF **decay gradually** (no clean cutoff), so you use **AIC/BIC** to select $p$ and $q$:

$$\text{AIC} = -2\ln(L) + 2k \qquad \text{BIC} = -2\ln(L) + k\ln(n)$$

Lower = better. BIC penalizes complexity more than AIC.

**Requires stationarity** — if data has trend, ARMA can't handle it. That's why we need ARIMA.

---


## 5.6 ARIMA(p, d, q)

**ARIMA = AR + I (Integration/Differencing) + MA**

$$\Phi(B)(1-B)^d Y_t = c + \Theta(B)\epsilon_t$$

ARIMA first **differences** the data $d$ times to make it stationary, then applies ARMA.

| Parameter | What | How to find |
|-----------|------|-------------|
| $p$ | AR order | PACF cutoff on differenced series |
| $d$ | Differencing order | ADF test (usually 0, 1, or 2) |
| $q$ | MA order | ACF cutoff on differenced series |

### Special Cases

| Model | ARIMA | Description |
|-------|-------|-------------|
| White Noise | (0,0,0) | No structure |
| Random Walk | (0,1,0) | $y_t = y_{t-1} + \epsilon$ |
| AR(1) | (1,0,0) | Pure autoregressive |
| MA(1) | (0,0,1) | Pure moving average |

### Python
`from statsmodels.tsa.arima.model import ARIMA`

---


In [ ]:
# 5.6 — Fit ARIMA on your data and forecast


## 5.7 SARIMA — Seasonal ARIMA

$$\text{SARIMA}(p,d,q)(P,D,Q)_m$$

Adds **seasonal terms** to ARIMA:

$$\Phi_P(B^m) \cdot \phi_p(B) \cdot (1-B)^d \cdot (1-B^m)^D \cdot Y_t = c + \Theta_Q(B^m) \cdot \theta_q(B) \cdot \epsilon_t$$

### Parameters

| | Non-Seasonal | Seasonal |
|--|-------------|----------|
| AR order | $p$ | $P$ (operates at lag $m, 2m, \ldots$) |
| Differencing | $d$ | $D$ (seasonal differencing) |
| MA order | $q$ | $Q$ (seasonal error terms) |
| Period | — | $m$ (12 for monthly, 7 for daily, etc.) |

### Finding Seasonal Orders

1. Apply $d$ regular + $D$ seasonal differencing
2. Plot ACF/PACF of the differenced series
3. Look at lags $m, 2m, 3m, \ldots$
   - ACF cuts off at seasonal lag $Qm$ → $Q$
   - PACF cuts off at seasonal lag $Pm$ → $P$

### Python
`from statsmodels.tsa.statespace.sarimax import SARIMAX`

---


In [ ]:
# 5.7 — Fit SARIMA and compare forecast with ARIMA


## 5.8 SARIMAX + Exogenous Variables

SARIMA**X** = SARIMA with e**X**ogenous (external) variables.

$$Y_t = \beta_1 X_{1,t} + \beta_2 X_{2,t} + \ldots + \text{SARIMA errors}$$

### What are Exogenous Variables?

External predictors that influence the target but are **not part of the target's own history**:
- Weather (temperature, rainfall) → affects ice cream sales
- Promotions/discounts → affects retail sales
- Economic indicators (interest rates) → affects housing prices
- Holiday calendar → affects traffic

### Important Caveat

At forecast time, you need **future values** of the exogenous variables. So they must be either:
- Known in advance (holidays, promotions you planned)
- Independently forecasted (weather forecast, economic projections)

---


In [ ]:
# 5.8 — Fit SARIMAX with month as an exogenous variable


## 5.9 Box-Jenkins Workflow

The **systematic methodology** for building ARIMA/SARIMA models:

### Step 1: Identification
1. Plot the series → check for trend, seasonality, changing variance
2. Transform if needed (log/Box-Cox for variance)
3. Difference to achieve stationarity → determine $d$ and $D$ (ADF/KPSS)
4. Plot ACF/PACF of differenced series → determine $p, q, P, Q$

### Step 2: Estimation
1. Fit the model (MLE)
2. Check that coefficients are significant

### Step 3: Diagnostic Checking
1. Residual plots (4-panel)
2. Ljung-Box test (p > 0.05)
3. If diagnostics fail → return to Step 1 with different orders

### Step 4: Forecasting
1. Generate point forecasts
2. Generate prediction intervals (widen with horizon)

```
Plot → Transform → Difference → ACF/PACF → Fit → Diagnose → Forecast
 ^                                                     |
 +-------- If diagnostics fail, revise model ----------+
```

---


## 5.10 Forecast + Confidence Intervals

A point forecast alone is not enough. You should always report **prediction intervals** showing the range of likely outcomes.

### Properties of Prediction Intervals

- They **widen** as the forecast horizon increases (more uncertainty)
- Typically 95% intervals: "we are 95% confident the true value falls within this range"
- Width depends on: residual variance, model complexity, forecast horizon

### Python
```python
pred = model.get_forecast(steps=24)
forecast = pred.predicted_mean
ci = pred.conf_int(alpha=0.05)  # 95% CI
```

---


In [ ]:
# 5.10 — Generate SARIMA forecast with confidence intervals and plot


# PART 6 — ML-Based Forecasting

---


## 6.1 Lag/Rolling Feature Engineering for ML

To use ML models (XGBoost, Random Forest), you need a **feature matrix**. Combine:

1. **Lag features**: `shift(1)`, `shift(2)`, `shift(12)`
2. **Rolling features**: `shift(1).rolling(w).mean()`, `.std()`
3. **Expanding features**: `shift(1).expanding().mean()`
4. **Calendar features**: month, quarter, sin/cos encodings
5. **Difference features**: `diff(1)`, `diff(12)`, `pct_change()`

### Golden Rule: Always `shift(1)` Before Rolling/Expanding

At prediction time for $y_t$, you only know up to $y_{t-1}$. So all features must use `.shift(1)` before any aggregation.

---


In [ ]:
# 6.1 — Build a complete feature engineering pipeline


## 6.2 XGBoost Forecasting

XGBoost (Extreme Gradient Boosting) is popular for time series because:
- Handles **non-linear** relationships
- Robust to outliers
- Captures **interactions** between features automatically
- Fast training
- Built-in **feature importance**

### Process
1. Create feature matrix (Section 6.1)
2. Split temporally (Section 3.6)
3. Fit `XGBRegressor` on training features → training target
4. Predict on test features
5. Evaluate (MAE, RMSE)

---


In [ ]:
# 6.2 — Train XGBoost, predict, and compute MAE


## 6.3 Feature Importance

Tree-based models provide **feature importance** — how much each feature contributes to predictions.

Types:
- **Gain-based**: Total reduction in loss attributed to the feature (default in XGBoost)
- **Permutation importance**: Drop in performance when the feature is randomly shuffled

### Why It Matters for Time Series

- Tells you which **lags** are most predictive (lag_1? lag_12?)
- Shows if **calendar features** (month) are useful
- Helps identify **useless features** to remove

---


In [ ]:
# 6.3 — Plot feature importance from your trained model


## 6.4 Leakage Prevention

**Data leakage** = your model accidentally uses future information during training. This makes results look great but the model fails in production.

### Common Leakage Sources in Time Series

| Leakage Type | Example | Fix |
|-------------|---------|-----|
| **Target leakage** | Rolling mean includes current value $y_t$ | Always `shift(1)` before rolling |
| **Future features** | Using tomorrow's weather to predict today | Only use features available at prediction time |
| **Test contamination** | Fitting scaler on full data including test | Fit scaler on train only |
| **Random split** | Future data in training set | Always split by time |

### How to Detect Leakage

- Results are **too good to be true** (e.g., R² = 0.999)
- Feature importance shows a suspicious feature dominating (e.g., `lag_0`)
- Walk-forward CV gives much worse results than simple train/test split

---


## 6.5 Walk-Forward ML Evaluation

Apply Walk-Forward CV (Section 3.7) to your ML model to get a robust performance estimate.

At each fold:
1. Train on expanding/sliding window of past data
2. Predict on the next test window
3. Compute metric (MAE)
4. Average across all folds

This simulates **real-world deployment** where you retrain periodically with new data.

---


In [ ]:
# 6.5 — Implement walk-forward ML evaluation loop


# PART 7 — Final Evaluation

---


## 7.1 MAE, RMSE, MAPE, MASE

### MAE (Mean Absolute Error)
$$\text{MAE} = \frac{1}{n} \sum |y_t - \hat{y}_t|$$
Simple, interpretable, same units as data.

### RMSE (Root Mean Squared Error)
$$\text{RMSE} = \sqrt{\frac{1}{n} \sum (y_t - \hat{y}_t)^2}$$
Penalizes large errors more. RMSE $\geq$ MAE always.

### MAPE (Mean Absolute Percentage Error)
$$\text{MAPE} = \frac{100}{n} \sum \left|\frac{y_t - \hat{y}_t}{y_t}\right|$$
Scale-free (percentage). But **undefined when $y_t = 0$** and **asymmetric** (penalizes under-forecasting more).

### MASE (Mean Absolute Scaled Error) — RECOMMENDED
$$\text{MASE} = \frac{\text{MAE}_{\text{model}}}{\text{MAE}_{\text{naive}}}$$

- MASE < 1 → model **beats** the naive baseline
- MASE = 1 → model equals naive
- MASE > 1 → model is **worse** than naive

Scale-free, handles zeros, symmetric. **Use this as your primary metric.**

---


In [ ]:
# 7.1 — Compute all 4 metrics for all your models


## 7.5 Model Comparison

Put all models in one table and compare:

```
| Model              | MAE   | RMSE  | MAPE  | MASE  |
|--------------------|-------|-------|-------|-------|
| Seasonal Naive     | ...   | ...   | ...   | 1.000 |
| ARIMA              | ...   | ...   | ...   | ...   |
| SARIMA             | ...   | ...   | ...   | ...   |
| XGBoost            | ...   | ...   | ...   | ...   |
```

Visualize with a bar chart.

---


In [ ]:
# 7.5 — Create comparison table and bar chart


## 7.6 Residual Analysis (Final)

For your **best model**, perform thorough residual analysis:

1. Plot residuals over time
2. ACF of residuals (should be white noise)
3. Histogram + Q-Q plot (check normality)
4. Ljung-Box test at multiple lags
5. Check for heteroscedasticity (changing variance in residuals)

If residuals pass all checks → your model has captured all available structure in the data.

---


In [ ]:
# 7.6 — Full residual analysis for your best model


# PART 8 — Stretch Topics

---


## 8.1 Facebook Prophet

$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

| Component | Description |
|-----------|-------------|
| $g(t)$ | Trend — piecewise linear or logistic growth with automatic **changepoints** |
| $s(t)$ | Seasonality — modeled using **Fourier series** at multiple frequencies |
| $h(t)$ | Holiday effects — user-specified calendar events |
| $\epsilon_t$ | Error term |

### Key Strengths
- Very easy to use (minimal configuration)
- Handles missing data and outliers gracefully
- Multiple seasonalities (daily + weekly + yearly)
- Built-in uncertainty intervals
- Automatic changepoint detection for trend shifts

### Installation
`pip install prophet`

### Prophet Expects
- DataFrame with two columns: `ds` (date) and `y` (value)

---


In [ ]:
# 8.1 — Fit Prophet and plot forecast with components


## 8.2 Auto-ARIMA (pmdarima)

Automates the Box-Jenkins identification step — searches over $(p,d,q) \times (P,D,Q)_m$ and selects the model with lowest AIC.

### How It Works
1. Uses ADF/KPSS to determine $d$ and $D$
2. Searches combinations of $p, q, P, Q$ using a **stepwise algorithm**
3. Selects model with lowest AIC/BIC

### Limitations
- Stepwise search may miss the global optimum
- Doesn't check residual diagnostics — you must do that manually

### Installation
`pip install pmdarima`

---


In [ ]:
# 8.2 — Run auto_arima and compare with your manual SARIMA


## 8.3 NeuralProphet

An extension of Prophet that adds **neural network components** (AR-Net) for capturing autocorrelation.

Combines the ease of Prophet with the modeling power of neural networks.

### Key Additions Over Prophet
- **AR component**: Learns autoregressive patterns using a neural network
- **Lagged regressors**: Use past values of exogenous features
- **PyTorch-based**: Can leverage GPU for training

### Installation
`pip install neuralprophet`

---


In [ ]:
# 8.3 — Fit NeuralProphet (if installed)


## 8.4 Deep Learning for Time Series — LSTM / N-BEATS / TFT

### LSTM (Long Short-Term Memory)
- A type of **Recurrent Neural Network** (RNN)
- Has "memory cells" that can learn to remember/forget patterns over long sequences
- Good for sequential data where long-term dependencies matter
- Challenge: requires careful normalization, windowing, and hyperparameter tuning
- Framework: PyTorch, TensorFlow/Keras

### N-BEATS (Neural Basis Expansion Analysis for Time Series)
- **Purely deep-learning** architecture designed specifically for time series
- Uses stacked fully-connected blocks with residual connections
- Two modes:
  - **Generic**: Learns any pattern (black-box)
  - **Interpretable**: Decomposes output into trend and seasonality (like STL but learned)
- Won the M4 forecasting competition

### TFT (Temporal Fusion Transformer)
- **Attention-based** architecture for multi-horizon forecasting
- Handles:
  - Static covariates (category of product)
  - Known future inputs (holidays, planned promotions)
  - Observed past inputs (past sales, past weather)
- Provides **interpretability** through attention weights and variable importance
- Best for complex, multi-variate forecasting problems

### When to Use Deep Learning

| Condition | Classical (ARIMA) | ML (XGBoost) | DL (LSTM/TFT) |
|-----------|------------------|-------------|----------------|
| Small data (< 1000 points) | Best | OK | Overfits |
| Medium data (1K-100K) | OK | Best | Good |
| Large data (100K+) | Slow | Good | Best |
| Many exogenous features | Limited | Good | Best (TFT) |
| Multiple related series | Per-series | Global model | Global model |
| Complex nonlinear patterns | Can't capture | Good | Best |

### Libraries
- **Darts** — unified API for ARIMA, Prophet, N-BEATS, TFT, etc.
- **NeuralForecast** — collection of DL forecasting models
- **PyTorch Forecasting** — TFT and DeepAR implementations

---

**END — Ab aap time series master hain. Code likho! 🚀**


In [ ]:
# 8.4 — Research and experiment with one DL model
